In [2]:
import requests
import io
import pandas
import numpy as np
#import matplotlib.pyplot as plt
from subprocess import Popen
import platform
import os
import shutil
from getpass import getpass
import netrc
from base64 import b64encode
from datetime import datetime

In [1]:
from tsgettoolbox import tsgettoolbox as tsget
from wdmtoolbox import wdmtoolbox as wdm


# Establish files required for API Access

In [3]:
# Set homeDir to current working directory
homeDir = os.path.expanduser("~/")
#homeDir = os.getcwd() + os.sep
# Create .urs_cookies and .dodsrc files in current directory
with open(homeDir + '.urs_cookies', 'w') as file:
    file.write('')
with open(homeDir + '.dodsrc', 'w') as file:
    file.write('HTTP.COOKIEJAR={}.urs_cookies\n'.format(homeDir))
    file.write('HTTP.NETRC={}.netrc'.format(homeDir))

print('Saved .urs_cookies and .dodsrc to:', homeDir)

# No need to copy .dodsrc since it's already in the working directory

Saved .urs_cookies and .dodsrc to: C:\Users\KVENABLE/


## Login with your Earthdata account. When completed your netrc file will be developed 

In [4]:
urs = 'urs.earthdata.nasa.gov'    # Earthdata URL to call for authentication
prompts = ['Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
           'Enter NASA Earthdata Login Password: ']

with open(homeDir + '.netrc', 'w') as file:
    file.write('machine {} login {} password {}'.format(urs, getpass(prompt=prompts[0]), getpass(prompt=prompts[1])))
    file.close()

print('Saved .netrc to:', homeDir)

# Set appropriate permissions for Linux/macOS
if platform.system() != "Windows":
    Popen('chmod og-rw ~/.netrc', shell=True)

Saved .netrc to: C:\Users\KVENABLE/


## Authenticate Credentials from login to generate token 

In [5]:
# Earthdata Login URL for obtaining the token, and creating one if it doesn't exist
url = 'https://urs.earthdata.nasa.gov/api/users/find_or_create_token'

# Earthdata Login credential prompts
#prompts = ['Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
 #          'Enter NASA Earthdata Login Password: ']

# Get credentials from user input
auth = netrc.netrc()
#auth = netrc.netrc(file=homeDir + '.netrc')
username, _, password = auth.authenticators('urs.earthdata.nasa.gov')

# Encode credentials using Base64
credentials = b64encode(f"{username}:{password}".encode('utf-8')).decode('utf-8')

# Headers with the Basic Authorization
headers = {
    'Authorization': f'Basic {credentials}'
}

# Make the POST request to get the token
response = requests.post(url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Parse the response JSON to get the token
    token_info = response.json()
    token = token_info.get("access_token")
    print("Token retrieved successfully")

    # Define the path for the .edl_token file in the home directory
    token_file_path = homeDir + ".edl_token"

    # Write the token to the .edl_token file
    with open(token_file_path, 'w') as token_file:
        token_file.write(token)

    print(f"Token saved to {token_file_path}")

else:
    print("Failed to retrieve token:", response.text)

Token retrieved successfully
Token saved to C:\Users\KVENABLE/.edl_token


In [20]:
def convertunitforHSPF(constituent, df, LDAS_var):
    '''This function is for unit conversion'''
    if constituent == "ATEMP": df = (df-273)
    #From K to degC
    if constituent == "SOLR": df = df * 0.0864
    #From W/m^2 to Langleys/day. 1 W/m^2 is 0.0864 Langleys/day
    if constituent == "PRECIP" and 'GLDAS' in LDAS_var:
        df = df * 3600
        #From kg/m^2/s to mm/hour
        #Assuming that 1 kg/m^2 is close to 1 mm
        df=df*3
            #GLDAS is 3 hourly data. THis changes values from
            #mm/hour to total precip for each times step of 3 hours.
    elif constituent == "PRECIP" and 'NLDAS' in LDAS_var:
        df=df
        #No change needed for NLDAS            
   
    return df


# Define the list of stations and constituents to download, and the details for each constituent

In [25]:
StationList=[['Seattle',47.629605, -122.348941,-8],
            ['Allahabad', 25.43,81.84,5.5],
            ['Asmara',15.33, 38.92,3]]


In [21]:
import pandas as pd
import numpy as np
import os
from tkinter import Tk, filedialog

# Function to validate latitude and longitude
def validate_lat_lon(lat, lon):
    try:
        lat = float(lat)
        lon = float(lon)
        if not (-90 <= lat <= 90):
            return False
        if not (-180 <= lon <= 180):
            return False
        return True
    except Exception:
        return False

# Prompt user for input method
print("How would you like to enter station data?")
print("1. Manual entry")
print("2. Upload from spreadsheet (CSV or XLSX)")
method = input("Enter 1 or 2: ").strip()

StationList = []

if method == '1':
    print("Enter station data as: StationName, Latitude, Longitude, TimeZoneAdjustment")
    print("Example: Seattle, 47.629605, -122.348941, -8")
    print("Enter a blank line to finish.")
    while True:
        entry = input("Station: ").strip()
        if not entry:
            break
        parts = [x.strip() for x in entry.split(',')]
        if len(parts) != 4:
            print("Invalid format. Please enter 4 values separated by commas.")
            continue
        name, lat, lon, tz = parts
        if not validate_lat_lon(lat, lon):
            print("Invalid latitude or longitude. Latitude must be -90 to 90, longitude -180 to 180.")
            continue
        try:
            StationList.append([name, float(lat), float(lon), float(tz)])
        except Exception:
            print("Invalid entry. Please check your values.")
elif method == '2':
    print("Please select your CSV or XLSX file with columns: StationName, Latitude, Longitude, TimeZoneAdjustment")
    Tk().withdraw()  # Hide the root window
    file_path = filedialog.askopenfilename(filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx;*.xls")])
    if file_path:
        if file_path.lower().endswith('.csv'):
            df = pd.read_csv(file_path)
        else:
            df = pd.read_excel(file_path)
        required_cols = ['StationName', 'Latitude', 'Longitude', 'TimeZoneAdjustment']
        if not all(col in df.columns for col in required_cols):
            print(f"Missing required columns. Found columns: {df.columns.tolist()}")
        else:
            for _, row in df.iterrows():
                if validate_lat_lon(row['Latitude'], row['Longitude']):
                    StationList.append([
                        str(row['StationName']),
                        float(row['Latitude']),
                        float(row['Longitude']),
                        float(row['TimeZoneAdjustment'])
                    ])
                else:
                    print(f"Invalid lat/lon for station {row['StationName']}. Skipping.")
    else:
        print("No file selected.")
else:
    print("Invalid selection. Please restart and choose 1 or 2.")

print("Final StationList:")
print(StationList)

How would you like to enter station data?
1. Manual entry
2. Upload from spreadsheet (CSV or XLSX)
Please select your CSV or XLSX file with columns: StationName, Latitude, Longitude, TimeZoneAdjustment
Final StationList:
[['CentralPark', 40.785091, -73.968285, -5.0], ['GoldenGate', 37.819929, -122.478255, -8.0], ['MillenniumPark', 41.882702, -87.619392, -6.0]]


In [22]:
from datetime import datetime

def get_date(prompt):
    while True:
        date_str = input(f"{prompt} (YYYY-MM-DD): ").strip()
        try:
            date_obj = datetime.strptime(date_str, "%Y-%m-%d")
            return date_obj.strftime("%Y-%m-%d")
        except ValueError:
            print("Invalid date format. Please enter the date as YYYY-MM-DD.")

print("Please enter the date range for data retrieval.")
start_date = get_date("Enter start date")
end_date = get_date("Enter end date")

print(f"Selected date range: {start_date} to {end_date}")

Please enter the date range for data retrieval.
Selected date range: 2018-01-30 to 2019-12-31


In [34]:
Constituent=['ATEMP','PRECIP', 'SOLR', 'WIND']
#Please refer to the GLDAS2 documentation below for more constituents.
#https://hydro1.gesdisc.eosdis.nasa.gov/data/GLDAS/README_GLDAS2.pdf
#https://hydro1.gesdisc.eosdis.nasa.gov/data/GLDAS/GLDAS_NOAH025_3H_EP.2.1/doc/README_GLDAS2.pdf


GLDAS_ConstituentDetails={
                    "PRECIP":["Precipitation","mm",
                    "GLDAS2:GLDAS_NOAH025_3H_2_1_Rainf_f_tavg","kg/m^2/s"],
                    "ATEMP":["Air Temperature","Fahrenheit",
                    "GLDAS2:GLDAS_NOAH025_3H_2_1_Tair_f_inst","K"],
                    "SOLR":["Shortwave radiation flux","Langleys",
                    "GLDAS2:GLDAS_NOAH025_3H_2_1_SWdown_f_tavg","W m-2"],
                    "WIND": ["Wind Speed", "m s-1",
                             "GLDAS2:GLDAS_NOAH025_3H_2_1_Wind_f_inst", "m s-1"] #average SW values for GLDAS. Instaneous is not available. I am using average values now.
                    }
#Please refer to the NLDAS documentation below for more constituents.
#https://hydro1.gesdisc.eosdis.nasa.gov/data/NLDAS/NLDAS2_README.pdf
NLDAS_ConstituentDetails={
                    "PRECIP":["Precipitation","mm",
                    "NLDAS:NLDAS_FORA0125_H_2_0_Rainf","mm"],
                    "ATEMP":["Air Temperature","Fahrenheit",
                    "NLDAS:NLDAS_FORA0125_H_2_0_Tair","K"],
                    "SOLR":["Shortwave radiation flux","Langleys",
                    "NLDAS:NLDAS_FORA0125_H_2_0_SWdown","W m-2"],
                    "WIND": ["Wind Speed", "m s-1",
                             "NLDAS:NLDAS_FORA0125_H_2_0_Wind_N",
                             "NLDAS:NLDAS_FORA0125_H_2_0_Wind_E", "m s-1"] #NLDAS provides wind speed in two components. I will need to calculate the resultant wind speed from these two components.
                    }

#If you add more constituents, you will need to expland this dict


In [30]:

UStop = 49.3457868 # north lat
USleft = -124.7844079 # west long
USright = -66.9513812 # east long
USbottom =  24.7433195 # south lat
#US Lat and long coordinates


In [35]:
#Testing after moving to that directory
WDMFileName='MetData_030926b.wdm'
wdm.createnewwdm(WDMFileName, overwrite=True)
index = 1
from datetime import datetime
with open("MetLog_030926b.txt", 'w') as Logfile:
    Logfile.write("Started Downloading the data at "
                + datetime.isoformat(datetime.now()) + " and saving in "
                + WDMFileName + "\n")
    for station in StationList:
        # Going through Each Station in the list
        TimeZoneAdjustment = station[3]
        Logfile.write("Station: " + station[0] + ", Latitude: " + str(station[1])
                        + ", Longitude: " + str(station[2])
                        + ", TimeZoneAdjustment: " + str(TimeZoneAdjustment)
                        + "\n")

        for const in Constituent:
            # Going through each constituent
            if station[1] < UStop and station[1] > USbottom and \
                station[2] > USleft and station[2] < USright:
                # NLDAS
                if const == "WIND":
                    wind_n_var = NLDAS_ConstituentDetails["WIND"][2]
                    wind_e_var = NLDAS_ConstituentDetails["WIND"][3]
                    TimeStep = 1
                    print(f'{station[0]} is in contiguous USA (NLDAS)')
                    print(f"Downloading WindN: {wind_n_var} and WindE: {wind_e_var}")
                    df_n = tsget.ldas(lat=station[1], lon=station[2],
                                      variables=wind_n_var,
                                      startDate=start_date,
                                      endDate=end_date)
                    df_e = tsget.ldas(lat=station[1], lon=station[2],
                                      variables=wind_e_var,
                                      startDate=start_date,
                                      endDate=end_date)
                    wind_speed = np.sqrt(df_n.iloc[:, 0] ** 2 + df_e.iloc[:, 0] ** 2)
                    wind_speed.name = "WIND"
                    df = wind_speed
                    column_name = "WIND"
                else:
                    LDAS_variable = NLDAS_ConstituentDetails[const][2]
                    TimeStep = 1
                    print(f'{station[0]} is in contiguous USA')
                    print(LDAS_variable)
                    df = tsget.ldas(lat=station[1], lon=station[2],
                                   variables=LDAS_variable,
                                   startDate=start_date,
                                   endDate=end_date)
                    column_name = df.columns[0]
                    df = df[column_name]
                    df.dropna()
                    df = convertunitforHSPF(const, df, LDAS_variable)
            else:
                # GLDAS
                if const == "WIND":
                    LDAS_variable = GLDAS_ConstituentDetails["WIND"][2]
                    TimeStep = 3
                    print(f'{station[0]} is outside contiguous USA (GLDAS)')
                    print(f"Downloading WIND: {LDAS_variable}")
                    df = tsget.ldas(lat=station[1], lon=station[2],
                                   variables=LDAS_variable,
                                   startDate=start_date,
                                   endDate=end_date)
                    column_name = df.columns[0]
                    df = df[column_name]
                    df.dropna()
                    df = convertunitforHSPF(const, df, LDAS_variable)
                else:
                    LDAS_variable = GLDAS_ConstituentDetails[const][2]
                    TimeStep = 3
                    print(f'{station} is outside contiguous USA')
                    print(LDAS_variable)
                    df = tsget.ldas(lat=station[1], lon=station[2],
                                   variables=LDAS_variable,
                                   startDate=start_date,
                                   endDate=end_date)
                    column_name = df.columns[0]
                    df = df[column_name]
                    df.dropna()
                    df = convertunitforHSPF(const, df, LDAS_variable)

            wdm.createnewdsn(WDMFileName, index,
                            constituent=const,
                            scenario="OBSERVED", location=station[0][0:8],
                            tcode=3, statid=station[0], tsstep=TimeStep,
                            description=LDAS_variable if const != "WIND" or station[1] >= UStop or station[1] <= USbottom or station[2] <= USleft or station[2] >= USright else "Vector Wind Speed")
            wdm.csvtowdm(WDMFileName, index, input_ts=df)
            Logfile.write("Constituent: " + const + ", Column Name:"
                + column_name + ", DSN: " + str(index) + "\n")
            index += 1

Seattle is in contiguous USA
NLDAS:NLDAS_FORA0125_H_2_0_Tair
Seattle is in contiguous USA
NLDAS:NLDAS_FORA0125_H_2_0_Rainf
Seattle is in contiguous USA
NLDAS:NLDAS_FORA0125_H_2_0_SWdown
Seattle is in contiguous USA (NLDAS)
['Allahabad', 25.43, 81.84, 5.5] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Tair_f_inst
['Allahabad', 25.43, 81.84, 5.5] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Rainf_f_tavg
['Allahabad', 25.43, 81.84, 5.5] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_SWdown_f_tavg
Allahabad is outside contiguous USA (GLDAS)
['Asmara', 15.33, 38.92, 3] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Tair_f_inst
['Asmara', 15.33, 38.92, 3] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_Rainf_f_tavg
['Asmara', 15.33, 38.92, 3] is outside contiguous USA
GLDAS2:GLDAS_NOAH025_3H_2_1_SWdown_f_tavg
Asmara is outside contiguous USA (GLDAS)


In [15]:
import os
from shutil import move

def move_to_cwd(filename):
    cwd = os.getcwd()
    dest = os.path.join(cwd, os.path.basename(filename))
    if os.path.abspath(filename) != os.path.abspath(dest):
        try:
            move(filename, dest)
            print(f"Moved {filename} to {dest}")
        except Exception as e:
            print(f"Could not move {filename}: {e}")
    else:
        print(f"{filename} is already in the current working directory.")

# Move WDM and MetLog files to current working directory after creation
move_to_cwd(WDMFileName)
move_to_cwd(Logfile.name)

MetData_030526.wdm is already in the current working directory.
MetLog_030526.txt is already in the current working directory.
